# Iteración 9 — Reentrenamiento reproducible del Tri-Ensamble (9 modelos) con persistencia de pesos

### ¿Qué se hizo?
Se reentrenó exactamente la configuración de la Iteración 8B (xlm-roberta-large, mdeberta-v3-base y cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual, 3 folds cada uno, mismos hiperparámetros, semillas y partición) en una GPU local RTX 4090, conservando los checkpoints del mejor epoch de cada fold en `outputs/` (pesos en fp16) junto con las probabilidades OOF y de evaluación.

### ¿Por qué?
En la Iteración 8B los checkpoints se eliminaban tras la inferencia para no llenar el disco de Kaggle, por lo que el modelo final no quedaba disponible como entregable ni era replicable. Con 24 GB de VRAM ya no se requiere gradient checkpointing, lo que reduce el tiempo de entrenamiento sin cambiar el tamaño de lote efectivo.

### Cambios frente a la Iteración 8B
* Rutas locales (`outputs/`) en lugar de `/kaggle/input` y `/kaggle/working`.
* Los 9 checkpoints se conservan (`outputs/<modelo>_fold<k>.pt`).
* Sin gradient checkpointing (mismo batch efectivo de 16).


In [ ]:
import os
import re
import gc
import sys
import glob
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
os.makedirs('outputs', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Dispositivo:", device, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# 1. Localizar y cargar datasets
train_path = glob.glob('**/train.csv', recursive=True)[0]
eval_path = glob.glob('**/eval.csv', recursive=True)[0]
train_df = pd.read_csv(train_path, encoding='utf-8')
eval_df = pd.read_csv(eval_path, encoding='utf-8')

label2id = {'negativo': 0, 'neutral': 1, 'positivo': 2}
id2label = {0: 'negativo', 1: 'neutral', 2: 'positivo'}
train_df['target'] = train_df['label'].map(label2id)

# 2. Detección de idioma para estratificación conjunta (clase + idioma)
def detect_lang(text):
    t = str(text).lower()
    words = set(re.findall(r'\b[a-záéíóúñãõçâêîôûàèù]+\b', t))
    es = len(words & {'el','la','los','las','que','en','un','una','por','para','con','pero','muy','lo','me','se','del','al','es','son','fue','está','más','también','cuando','todo','esta','este','bien','malo','bueno','llegó','compré','pedí','uso','días','casa'})
    en = len(words & {'the','and','to','of','is','in','it','for','that','with','on','was','but','not','are','have','this','my','very','good','great','bad','product','bought','use','days','quality','battery','would','after'})
    pt = len(words & {'não','com','uma','para','por','mais','como','mas','muito','bem','bom','boa','ruim','comprei','chegou','dias','casa','uso','também','quando','isso','essa','esse','são','está','fazer','qualidade','bateria','nas','dos','das','pelo','pela'})
    fr = len(words & {'le','la','les','des','est','une','dans','pour','pas','que','avec','sur','mais','plus','très','bien','bon','mauvais','acheté','livraison','jours','maison','qualité','batterie','depuis','tout','cette','sont','fait','avoir'})
    if es >= 3 and en >= 3:
        return 'es-en'
    scores = {'es': es, 'en': en, 'pt': pt, 'fr': fr}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'other'

train_df['lang'] = train_df['text'].apply(detect_lang)
train_df['strat_key'] = train_df['label'] + '_' + train_df['lang']

# Agrupar combinaciones extremadamente raras (< 3 muestras) para evitar advertencias en StratifiedKFold(n_splits=3)
vc = train_df['strat_key'].value_counts()
rare_keys = vc[vc < 3].index
train_df.loc[train_df['strat_key'].isin(rare_keys), 'strat_key'] = train_df['label'] + '_rare'

class ReviewDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=160, labels=None):
        self.enc = tokenizer(
            list(texts), truncation=True, padding='max_length', max_length=max_len, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.enc['input_ids'])

    def __getitem__(self, idx):
        item = {'input_ids': self.enc['input_ids'][idx], 'attention_mask': self.enc['attention_mask'][idx]}
        if self.labels is not None:
            item['labels'] = self.labels[idx]
        return item

def clear_vram(*objs):
    sys.last_traceback = None
    sys.last_value = None
    for obj in objs:
        del obj
    gc.collect()
    torch.cuda.empty_cache()

def run_kfold_cv(model_name, short_name, n_splits=3, epochs=4, batch_size=8, grad_accum=2, lr=1e-5, use_fp16=True):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_ds = ReviewDataset(eval_df['text'], tokenizer, max_len=160)
    eval_loader = DataLoader(eval_ds, batch_size=32, shuffle=False)

    oof_probs = np.zeros((len(train_df), 3), dtype=np.float32)
    eval_probs_folds = np.zeros((n_splits, len(eval_df), 3), dtype=np.float32)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (trn_idx, val_idx) in enumerate(skf.split(train_df, train_df['strat_key'])):
        print(f"\n{'='*55}\n[{short_name}] Fold {fold+1}/{n_splits} | Train: {len(trn_idx)} | Val: {len(val_idx)}\n{'='*55}")
        seed_everything(42 + fold)

        trn_ds = ReviewDataset(train_df.iloc[trn_idx]['text'], tokenizer, labels=train_df.iloc[trn_idx]['target'].values)
        val_ds = ReviewDataset(train_df.iloc[val_idx]['text'], tokenizer, labels=train_df.iloc[val_idx]['target'].values)

        trn_loader = DataLoader(trn_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
        if not use_fp16:
            model = model.float()  # Necesario para DeBERTa-v3 para evitar overflow numérico
        model = model.to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = (len(trn_loader) // grad_accum) * epochs
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
        scaler = torch.amp.GradScaler('cuda', enabled=(use_fp16 and torch.cuda.is_available()))

        best_val_acc = 0.0
        best_val_probs = None
        ckpt_path = f'outputs/{short_name}_fold{fold}.pt'

        for epoch in range(epochs):
            model.train()
            optimizer.zero_grad()
            for step, batch in enumerate(trn_loader):
                if use_fp16:
                    with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                        loss = model(
                            input_ids=batch['input_ids'].to(device),
                            attention_mask=batch['attention_mask'].to(device),
                            labels=batch['labels'].to(device)
                        ).loss / grad_accum
                    scaler.scale(loss).backward()
                    if (step + 1) % grad_accum == 0 or (step + 1) == len(trn_loader):
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer)
                        scaler.update()
                        optimizer.zero_grad()
                        scheduler.step()
                else:
                    loss = model(
                        input_ids=batch['input_ids'].to(device),
                        attention_mask=batch['attention_mask'].to(device),
                        labels=batch['labels'].to(device)
                    ).loss / grad_accum
                    loss.backward()
                    if (step + 1) % grad_accum == 0 or (step + 1) == len(trn_loader):
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                        optimizer.zero_grad()
                        scheduler.step()

            model.eval()
            v_probs = []
            with torch.no_grad():
                for batch in val_loader:
                    logits = model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)).logits
                    v_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
            v_probs = np.vstack(v_probs)
            acc = accuracy_score(train_df.iloc[val_idx]['target'].values, np.argmax(v_probs, axis=1))
            print(f"  Época {epoch+1}/{epochs} | Val Accuracy: {acc:.4f}")

            if acc > best_val_acc:
                best_val_acc = acc
                best_val_probs = v_probs
                torch.save({k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()}, ckpt_path)

        oof_probs[val_idx] = best_val_probs

        # Inferencia sobre eval.csv usando el mejor checkpoint del fold
        model.load_state_dict(torch.load(ckpt_path))
        model.eval()
        e_probs = []
        with torch.no_grad():
            for batch in eval_loader:
                logits = model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)).logits
                e_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        eval_probs_folds[fold] = np.vstack(e_probs)

        clear_vram(model, optimizer, scheduler, scaler)

    oof_acc = accuracy_score(train_df['target'].values, np.argmax(oof_probs, axis=1))
    print(f"\n>>> [{short_name}] OOF Accuracy Global (3-Fold): {oof_acc:.4f}")

    mean_eval_probs = eval_probs_folds.mean(axis=0)
    np.save(f'outputs/oof_{short_name}.npy', oof_probs)
    np.save(f'outputs/eval_{short_name}.npy', mean_eval_probs)
    return oof_probs, mean_eval_probs

In [ ]:
oof_xlmr, eval_xlmr = run_kfold_cv(
    model_name='xlm-roberta-large',
    short_name='xlmr_3f',
    n_splits=3,
    epochs=4,
    batch_size=8,
    grad_accum=2,      # Batch efectivo = 16
    lr=1e-5,
    use_fp16=True
)

In [ ]:
oof_mdeberta, eval_mdeberta = run_kfold_cv(
    model_name='microsoft/mdeberta-v3-base',
    short_name='mdeberta_3f',
    n_splits=3,
    epochs=4,
    batch_size=16,
    grad_accum=1,      # Batch efectivo = 16
    lr=2e-5,
    use_fp16=False     # fp32 obligatorio para evitar ValueError/overflow en DeBERTa-v3
)

##2. Entrenamiento 3-Fold de la Tercera Arquitectura (cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual)
###¿Qué se hizo?
Se invocó el motor run_kfold_cv exactamente sobre los mismos 3 pliegues estratificados (n_splits=3, random_state=42, llave label + lang) para entrenar cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual durante 4 épocas (batch_size=16, lr=2e-5, FP16), alcanzando un OOF Accuracy individual de 0.9015 (Fold 1: 0.9025, Fold 2: 0.8984, Fold 3: 0.9037) y serializando oof_cardiff_3f.npy y eval_cardiff_3f.npy.
###¿Por qué se decidió hacer eso?
Aunque XLM-RoBERTa-Large (0.9095) y mDeBERTa-v3-Base (0.9059) tienen alta capacidad general, ambos fueron preentrenados exclusivamente con objetivos autosupervisados sobre texto enciclopédico y web general (Wikipedia/CommonCrawl).CardiffNLP-XLM-R-Sentiment aporta diversidad de dominio y preentrenamiento afectivo, al haber sido entrenado específicamente sobre millones de opiniones reales multilingües, expresiones coloquiales, sarcasmo y mezcla de idiomas (code-switching), cometiendo errores poco correlacionados con los dos modelos base.   
### ¿Qué otras alternativas había y por qué no se hicieron?
* Entrenar microsoft/mdeberta-v3-large: Aunque es más grande, requería FP32 obligatorio (como se comprobó en la Iteración 7) y más de 3.5 horas de GPU para los 3 folds, superando el límite de tiempo por sesión en Kaggle.
* Añadir un modelo clásico TF-IDF + Regresión Logística: Su accuracy individual (~0.83) estaba demasiado por debajo de los Transformers (0.901–0.909), corriendo el riesgo de diluir la señal semántica en reseñas con negaciones o ironía.
###¿Qué cambia frente al código anterior (Iteración 7)?
Se amplía el repertorio de modelos entrenados de 2 a 3 familias distintas (9 modelos entrenados en total a través de los 3 folds), reutilizando la misma función run_kfold_cv y garantizando alineación exacta en las 20,466 filas OOF.

In [ ]:
oof_xlmr = np.load('outputs/oof_xlmr_3f.npy')
oof_mdeberta = np.load('outputs/oof_mdeberta_3f.npy')
eval_xlmr = np.load('outputs/eval_xlmr_3f.npy')
eval_mdeberta = np.load('outputs/eval_mdeberta_3f.npy')

y_true = train_df['target'].values

print("=== Desempeño Out-Of-Fold (100% de train.csv, n=20,466) ===")
print(f"1. XLM-RoBERTa-Large (3-Fold): {accuracy_score(y_true, np.argmax(oof_xlmr, axis=1)):.4f}")
print(f"2. mDeBERTa-v3-Base  (3-Fold): {accuracy_score(y_true, np.argmax(oof_mdeberta, axis=1)):.4f}")

# Búsqueda fina de pesos óptimos (w_xlmr + w_mdeberta = 1.0)
best_w = 0.55
best_oof_acc = 0.0

for w in np.linspace(0.30, 0.75, 46):
    blend = w * oof_xlmr + (1.0 - w) * oof_mdeberta
    acc = accuracy_score(y_true, np.argmax(blend, axis=1))
    if acc > best_oof_acc:
        best_oof_acc = acc
        best_w = round(float(w), 2)

w_xlmr = best_w
w_mdeb = round(1.0 - w_xlmr, 2)

oof_ens = w_xlmr * oof_xlmr + w_mdeb * oof_mdeberta
oof_preds = np.argmax(oof_ens, axis=1)

print(f"\n=== Ensamble 3-Fold Soft Voting Óptimo (XLM-R={w_xlmr}, mDeBERTa={w_mdeb}) ===")
print(f"OOF Accuracy Ensamble (20,466 muestras): {best_oof_acc:.4f}")
print(classification_report(y_true, oof_preds, target_names=['negativo', 'neutral', 'positivo'], digits=4))

# Aplicar pesos óptimos sobre eval.csv
eval_ens = w_xlmr * eval_xlmr + w_mdeb * eval_mdeberta
eval_preds = np.argmax(eval_ens, axis=1)

sub_7 = pd.DataFrame({
    'id': eval_df['id'],
    'answer': [id2label[p] for p in eval_preds]
})
sub_7.to_csv('outputs/submission_7_kfold_ensemble.csv', index=False)

print("\nArchivo guardado en outputs/submission_7_kfold_ensemble.csv")
print("Distribución de predicciones en eval.csv:")
print(sub_7['answer'].value_counts(normalize=True).round(4))

##4. Optimización del Tri-Ensamble (9 Modelos), Selección Automática de Calibración y Exportación (submission_8b_tri_ensemble.csv)
###¿Qué se hizo?

* Se cargaron los 6 tensores .npy correspondientes a las 3 arquitecturas (oof y eval de xlmr_3f, mdeberta_3f y cardiff_3f).

* Se ejecutó una búsqueda en grilla sobre el símplex convexo w1+w2+w3=1, encontrando la combinación óptima en XLM-R-Large = 0.50, mDeBERTa-v3-Base = 0.20, CardiffNLP-XLM-R = 0.30, elevando el OOF Accuracy Global a 0.9114 y el Macro F1 a 0.9185 (neutral F1: 0.9988, positivo F1: 0.8813, negativo F1: 0.8753).   
baseline8.ipynb

* Se aplicó nuevamente la verificación de calibración anidada de 5 pliegues (0.9109 calibrado vs. 0.9114 sin calibrar) y se programó una regla de decisión automática (use_cal = cal_oof_acc > best_oof_acc) que descartó los multiplicadores y exportó el tri-ensamble puro en outputs/submission_8b_tri_ensemble.csv (positivo: 43.19%, negativo: 37.71%, neutral: 19.10%).   
baseline8.ipynb

###¿Por qué se decidió hacer eso?

* Efecto de rescate por consenso de 3 familias: A diferencia de un ensamble de 2 modelos (donde una discrepancia depende exclusivamente de la magnitud de probabilidad), con 3 arquitecturas distintas (0.50 / 0.20 / 0.30) cuando XLM-R-Large duda en una reseña coloquial o irónica, la coincidencia entre CardiffNLP (0.30) y mDeBERTa (0.20) suma 0.50 de peso conjunto y corrige la predicción, explicando por qué CardiffNLP (a pesar de tener 0.9015 individual) recibe un peso mayor (0.30) que mDeBERTa (0.20, con 0.9059 individual).   

* Compuerta de seguridad anti-sobreajuste (use_cal): Automatizar la comparación OOF antes de exportar garantiza que ninguna técnica de post-procesamiento se aplique a ciegas si no demuestra ganancia real fuera de muestra.   
baseline8.ipynb

###¿Qué otras alternativas había y por qué no se hicieron?

* Asignar pesos proporcionales al accuracy individual (ej. 0.45 / 0.35 / 0.20): Se descartó en favor de la optimización empírica sobre OOF, que demostró que CardiffNLP aporta mayor complementariedad marginal (0.30) que su ranking individual sugeriría.   

* Promediar rangos (Rank Averaging) en lugar de probabilidades (Soft Voting): Se descartó porque los 3 modelos comparten exactamente la misma función de pérdida (entropía cruzada multiclase con softmax), por lo que sus escalas probabilísticas ya son directamente comparables.   


###¿Qué cambia frente al código anterior (Iteración 7)?

Se pasa de un bi-ensamble de 6 modelos (0.9109 OOF) a un tri-ensamble heterogéneo de 9 modelos (0.9114 OOF), incorporando optimización de pesos en 3 dimensiones, validación cruzada anidada de umbrales y selección automática basada en evidencia OOF para generar el envío definitivo

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

oof_xlmr = np.load('outputs/oof_xlmr_3f.npy')
oof_mdeberta = np.load('outputs/oof_mdeberta_3f.npy')
eval_xlmr = np.load('outputs/eval_xlmr_3f.npy')
eval_mdeberta = np.load('outputs/eval_mdeberta_3f.npy')

y_true = train_df['target'].values
oof_ens = 0.50 * oof_xlmr + 0.50 * oof_mdeberta
eval_ens = 0.50 * eval_xlmr + 0.50 * eval_mdeberta

# Optimización de multiplicadores de clase con validación cruzada anidada (evita sobreajuste)
nested_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
oof_cal_preds = np.zeros_like(y_true)
fold_mults = []

for tr_i, va_i in nested_skf.split(oof_ens, y_true):
    def loss_func(weights):
        scaled = oof_ens[tr_i] * weights
        return -accuracy_score(y_true[tr_i], np.argmax(scaled, axis=1))

    res = minimize(
        loss_func,
        x0=[1.0, 1.0, 1.0],
        method='Nelder-Mead',
        options={'maxiter': 500, 'xatol': 1e-4}
    )
    w_opt = res.x / res.x[1]  # Normalizar respecto a la clase neutral
    fold_mults.append(w_opt)
    oof_cal_preds[va_i] = np.argmax(oof_ens[va_i] * w_opt, axis=1)

mean_mults = np.mean(fold_mults, axis=0)
print(f"Multiplicadores promedio aprendidos [negativo, neutral, positivo]: {np.round(mean_mults, 4)}")
print(f"OOF Accuracy Sin Calibrar: {accuracy_score(y_true, np.argmax(oof_ens, axis=1)):.4f}")
print(f"OOF Accuracy Calibrado:    {accuracy_score(y_true, oof_cal_preds):.4f}")

# Generar submission_8a_calibrated.csv
eval_cal_preds = np.argmax(eval_ens * mean_mults, axis=1)
sub_8a = pd.DataFrame({
    'id': eval_df['id'],
    'answer': [id2label[p] for p in eval_cal_preds]
})
sub_8a.to_csv('outputs/submission_8a_calibrated.csv', index=False)
print("\nDistribución eval.csv (Calibrado):")
print(sub_8a['answer'].value_counts(normalize=True).round(4))

In [ ]:
oof_cardiff, eval_cardiff = run_kfold_cv(
    model_name='cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual',
    short_name='cardiff_3f',
    n_splits=3,
    epochs=4,
    batch_size=16,
    grad_accum=1,
    lr=2e-5,
    use_fp16=True
)

In [ ]:
oof_xlmr = np.load('outputs/oof_xlmr_3f.npy')
oof_mdeberta = np.load('outputs/oof_mdeberta_3f.npy')
oof_cardiff = np.load('outputs/oof_cardiff_3f.npy')

eval_xlmr = np.load('outputs/eval_xlmr_3f.npy')
eval_mdeberta = np.load('outputs/eval_mdeberta_3f.npy')
eval_cardiff = np.load('outputs/eval_cardiff_3f.npy')

y_true = train_df['target'].values

print("=== Desempeño Out-Of-Fold Individual (n=20,466) ===")
print(f"1. XLM-RoBERTa-Large (3-Fold):       {accuracy_score(y_true, np.argmax(oof_xlmr, axis=1)):.4f}")
print(f"2. mDeBERTa-v3-Base  (3-Fold):       {accuracy_score(y_true, np.argmax(oof_mdeberta, axis=1)):.4f}")
print(f"3. CardiffNLP-XLM-R  (3-Fold):       {accuracy_score(y_true, np.argmax(oof_cardiff, axis=1)):.4f}")

# Búsqueda en grilla de pesos óptimos (w1 + w2 + w3 = 1)
best_w = (0.40, 0.35, 0.25)
best_oof_acc = 0.0

for w1 in np.linspace(0.25, 0.55, 13):
    for w2 in np.linspace(0.20, 0.50, 13):
        w3 = round(1.0 - w1 - w2, 4)
        if w3 < 0.05 or w3 > 0.45:
            continue
        blend = w1 * oof_xlmr + w2 * oof_mdeberta + w3 * oof_cardiff
        acc = accuracy_score(y_true, np.argmax(blend, axis=1))
        if acc > best_oof_acc:
            best_oof_acc = acc
            best_w = (round(float(w1), 3), round(float(w2), 3), round(float(w3), 3))

w1, w2, w3 = best_w
oof_tri = w1 * oof_xlmr + w2 * oof_mdeberta + w3 * oof_cardiff
eval_tri = w1 * eval_xlmr + w2 * eval_mdeberta + w3 * eval_cardiff

print(f"\n=== Tri-Ensamble Soft Voting (XLM-R={w1}, mDeBERTa={w2}, Cardiff={w3}) ===")
print(f"OOF Accuracy Tri-Ensamble: {best_oof_acc:.4f}")

# Calibración de umbrales anidada sobre el Tri-Ensamble
nested_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
oof_tri_cal = np.zeros_like(y_true)
mults_list = []

for tr_i, va_i in nested_skf.split(oof_tri, y_true):
    res = minimize(
        lambda m: -accuracy_score(y_true[tr_i], np.argmax(oof_tri[tr_i] * m, axis=1)),
        x0=[1.0, 1.0, 1.0],
        method='Nelder-Mead',
        options={'maxiter': 500, 'xatol': 1e-4}
    )
    m_opt = res.x / res.x[1]
    mults_list.append(m_opt)
    oof_tri_cal[va_i] = np.argmax(oof_tri[va_i] * m_opt, axis=1)

final_mults = np.mean(mults_list, axis=0)
cal_oof_acc = accuracy_score(y_true, oof_tri_cal)

print(f"Multiplicadores finales [neg, neu, pos]: {np.round(final_mults, 4)}")
print(f"OOF Accuracy Tri-Ensamble + Calibración: {cal_oof_acc:.4f}")

# Elegir automáticamente si aplicar calibración solo si mejora el OOF anidado
use_cal = cal_oof_acc > best_oof_acc
final_eval_probs = eval_tri * final_mults if use_cal else eval_tri
final_oof_preds = oof_tri_cal if use_cal else np.argmax(oof_tri, axis=1)

print(f"Estrategia seleccionada para envío: {'Tri-Ensamble + Calibración' if use_cal else 'Tri-Ensamble Sin Calibrar'}")
print(classification_report(y_true, final_oof_preds, target_names=['negativo', 'neutral', 'positivo'], digits=4))

sub_8b = pd.DataFrame({
    'id': eval_df['id'],
    'answer': [id2label[p] for p in np.argmax(final_eval_probs, axis=1)]
})
sub_8b.to_csv('outputs/submission_8b_tri_ensemble.csv', index=False)
print("\nArchivo guardado en outputs/submission_8b_tri_ensemble.csv")
print("Distribución eval.csv:")
print(sub_8b['answer'].value_counts(normalize=True).round(4))